Importaciones necesarias

# Creación de la base de datos

In [8]:
import warnings
warnings.filterwarnings('ignore')
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = "postgres"  # Base existente para crear otras bases

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(
    DATABASE_URL,
    isolation_level="AUTOCOMMIT"
)

try:
    with engine.connect() as connection:
        connection.execute(text("CREATE DATABASE clip_pay"))
        print("Base de datos creada exitosamente.")
except Exception as e:
    print(f"No se pudo crear la base de datos: {e}")
finally:
    engine.dispose()

Base de datos creada exitosamente.


Creación de tablas

In [9]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy.orm import sessionmaker, declarative_base

# Cargar variables del archivo .env
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_NAME = "clip_pay"

# Crear URL de conexión
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

# Conexión a PostgreSQL
engine = create_engine(DATABASE_URL)

# Crear sesiones
Session = sessionmaker(bind=engine)
session = Session()

# Clase base para modelos ORM
Base = declarative_base()

In [10]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    DateTime,
    ForeignKey,
    CheckConstraint
)

class Customer(Base):
    __tablename__ = 'customers'

    customer_id = Column(Integer, primary_key=True)

    age = Column(Integer, nullable=False)

    city = Column(String(100), nullable=False)

    # Restricciones para validar datos
    __table_args__ = (
        CheckConstraint(
            "age >= 18",
            name="ck_customer_age"
        ),
    )


class Merchant(Base):
    __tablename__ = 'merchants'

    merchant_id = Column(Integer, primary_key=True)

    merchant_name = Column(String(100), nullable=False)

    category = Column(String(100), nullable=False)

    # Restricciones para validar datos
    __table_args__ = (
        CheckConstraint(
            "length(merchant_name) > 0",
            name="ck_merchant_name_no_vacio"
        ),
    )


class PaymentMethod(Base):
    __tablename__ = 'payment_methods'

    payment_method_id = Column(Integer, primary_key=True)

    customer_id = Column(
        Integer,
        ForeignKey('customers.customer_id'),
        nullable=False
    )

    method_type = Column(String(50), nullable=False)

    # Restricciones para validar datos
    __table_args__ = (
        CheckConstraint(
            "length(method_type) > 0",
            name="ck_payment_method_type_no_vacio"
        ),
    )


class Transaction(Base):
    __tablename__ = 'transactions'

    transaction_id = Column(Integer, primary_key=True)

    customer_id = Column(
        Integer,
        ForeignKey('customers.customer_id'),
        nullable=False
    )

    merchant_id = Column(
        Integer,
        ForeignKey('merchants.merchant_id'),
        nullable=False
    )

    payment_method_id = Column(
        Integer,
        ForeignKey('payment_methods.payment_method_id'),
        nullable=False
    )

    amount = Column(Float, nullable=False)

    transaction_date = Column(DateTime, nullable=False)

    # Restricciones para validar datos
    __table_args__ = (
        CheckConstraint(
            "amount > 0",
            name="ck_transaction_amount_positive"
        ),
    )


Base.metadata.create_all(engine)
print("Tablas creadas correctamente.")

Tablas creadas correctamente.


Llenado de base de datos

In [11]:
import pandas as pd

# Conexión a PostgreSQL
connection = engine.raw_connection()


# Leer los archivos CSV
customers = pd.read_csv("customers.csv")
merchants = pd.read_csv("merchants.csv")
payment_methods = pd.read_csv("payment_methods.csv")
transactions = pd.read_csv("transactions.csv")


# Mostrar los datos antes de insertarlos
print("Customers:")
print(customers.head())

print("\nMerchants:")
print(merchants.head())

print("\nPayment Methods:")
print(payment_methods.head())

print("\nTransactions:")
print(transactions.head())


# Insertar los datos en PostgreSQL
customers.to_sql(
    "customers",
    engine,
    if_exists="append",
    index=False
)

merchants.to_sql(
    "merchants",
    engine,
    if_exists="append",
    index=False
)

payment_methods.to_sql(
    "payment_methods",
    engine,
    if_exists="append",
    index=False
)

transactions.to_sql(
    "transactions",
    engine,
    if_exists="append",
    index=False
)


print("\nDatos cargados correctamente.")

Customers:
   customer_id  age              city
0            1   58            Mérida
1            2   25         Monterrey
2            3   19           Tijuana
3            4   65   San Luis Potosí
4            5   35  Ciudad de México

Merchants:
   merchant_id merchant_name                category
0            1       Walmart            Supermercado
1            2       Soriana            Supermercado
2            3      Chedraui            Supermercado
3            4          OXXO  Tienda de conveniencia
4            5      7-Eleven  Tienda de conveniencia

Payment Methods:
   payment_method_id  customer_id         method_type
0                  1            1  Tarjeta de crédito
1                  2            2  Tarjeta de crédito
2                  3            2  Transferencia SPEI
3                  4            3   Billetera digital
4                  5            3  Transferencia SPEI

Transactions:
   transaction_id  customer_id  merchant_id  payment_method_id  amount  \


# Ejemplos de Consultas

PARA VOLVER A HACER CONSULTAS
Ejecuta:

In [44]:
from database import engine
import pandas as pd

connection = engine.raw_connection()

print("Conectado a clip_pay")

Conectado a clip_pay


SELECT

In [45]:
connection.rollback()

print("Todo sobre los tres primeros clientes")
pd.read_sql(
    'SELECT * FROM customers ' \
    'LIMIT 3', 
    connection
    )

Todo sobre los tres primeros clientes


,customer_id,age,city
0,1,58,Mérida
1,2,25,Monterrey
2,3,19,Tijuana


WHERE

In [16]:
connection.rollback()

print("Clientes mayores de 30 años y que sean de CDMX")
pd.read_sql(
    "SELECT * FROM customers " \
    "WHERE age >= 30 AND city = 'Ciudad de México'",
    connection
    )

Clientes mayores de 30 años y que sean de CDMX


,customer_id,age,city
0,5,35,Ciudad de México
1,11,61,Ciudad de México
2,14,52,Ciudad de México
3,23,50,Ciudad de México
4,30,62,Ciudad de México
...,...,...,...
133,974,45,Ciudad de México
134,976,31,Ciudad de México
135,982,34,Ciudad de México
136,992,30,Ciudad de México


GROUP BY

In [17]:
connection.rollback()

print("Número de clientes por ciudad")
pd.read_sql(
    "SELECT city, " \
    "COUNT(*) AS total_customers " \
    "FROM customers " \
    "GROUP BY city",
    connection
    )

Número de clientes por ciudad


,city,total_customers
0,Mérida,50
1,San Luis Potosí,37
2,Puebla,66
3,Aguascalientes,25
4,Toluca,53
5,Saltillo,24
6,Guadalajara,138
7,Morelia,31
8,Chihuahua,27
9,Ciudad de México,198


In [18]:
connection.rollback()

print("Ventas por cliente")
pd.read_sql(
    "SELECT customer_id, " \
    "SUM(amount) AS total_spent " \
    "FROM transactions " \
    "GROUP BY customer_id " \
    "LIMIT 3",
    connection
    )

Ventas por cliente


,customer_id,total_spent
0,652,1129.37
1,273,9550.00
2,51,1993.83


ORDER BY

In [19]:
connection.rollback()

print("Clientes de mayor a menor edad")
pd.read_sql(
    "SELECT * FROM customers " \
    "ORDER BY age DESC;",
    connection
    )

Clientes de mayor a menor edad


,customer_id,age,city
0,562,80,Monterrey
1,100,80,León
2,611,80,León
3,480,80,Ciudad de México
4,348,79,Monterrey
...,...,...,...
995,919,18,Ciudad de México
996,870,18,Puebla
997,443,18,Guadalajara
998,923,18,Monterrey


In [20]:
connection.rollback()

print("Suma de gastos por cliente ordenados de mayor a menor")
pd.read_sql(
    "SELECT customer_id, SUM(amount) AS total_spent " \
    "FROM transactions " \
    "GROUP BY customer_id " \
    "ORDER BY total_spent DESC;",
    connection
    )

Suma de gastos por cliente ordenados de mayor a menor


,customer_id,total_spent
0,28,258421.31
1,775,189292.23
2,235,125002.64
3,577,110575.15
4,452,88460.02
...,...,...
995,731,432.10
996,974,408.76
997,386,182.22
998,546,165.81


HAVING

In [21]:
connection.rollback()

print("Clientes que han gastado más de $5,000")
pd.read_sql(
    "SELECT " \
    "customer_id, " \
    "SUM(amount) AS total_spent " \
    "FROM transactions " \
    "GROUP BY customer_id " \
    "HAVING SUM(amount) > 5000;",
    connection
    )

Clientes que han gastado más de $5,000


,customer_id,total_spent
0,273,9550.00
1,951,7052.16
2,70,56413.09
3,350,9460.17
4,176,5565.47
...,...,...
528,465,5235.42
529,449,11089.99
530,148,6853.95
531,790,23986.95


JOIN - combina información de varias tablas.

In [22]:
connection.rollback()

print("Transacciones con el id & ciudad del cliente y el nombre del comercio")
pd.read_sql(
    "SELECT " \
    "t.transaction_id, " \
    "c.customer_id, " \
    "c.city, " \
    "m.merchant_name, " \
    "t.amount " \
    "FROM transactions t " \
    "JOIN customers c " \
    "ON t.customer_id = c.customer_id " \
    "JOIN merchants m " \
    "ON t.merchant_id = m.merchant_id;",
    connection
    )

Transacciones con el id & ciudad del cliente y el nombre del comercio


,transaction_id,customer_id,city,merchant_name,amount
0,1,917,Toluca,Walmart,177.94
1,2,383,Mérida,Shein,983.66
2,3,359,Cancún,7-Eleven,128.32
3,4,595,Monterrey,Costco,705.96
4,5,909,Querétaro,Cinemex,164.38
...,...,...,...,...,...
24995,24996,611,León,Uber,218.36
24996,24997,547,León,Circle K,46.10
24997,24998,678,Morelia,7-Eleven,71.72
24998,24999,222,Toluca,Amazon México,241.21


INNER JOIN - devuelve solamente los registros que tienen coincidencia en ambas tablas.

In [23]:
connection.rollback()

print("Transacciones por cliente, solo aparecen clientes con transacciones")
pd.read_sql(
    "SELECT " \
    "c.customer_id, " \
    "c.city, " \
    "t.amount " \
    "FROM customers c " \
    "INNER JOIN transactions t " \
    "ON c.customer_id = t.customer_id;",
    connection
    )

Transacciones por cliente, solo aparecen clientes con transacciones


,customer_id,city,amount
0,917,Toluca,177.94
1,383,Mérida,983.66
2,359,Cancún,128.32
3,595,Monterrey,705.96
4,909,Querétaro,164.38
...,...,...,...
24995,611,León,218.36
24996,547,León,46.10
24997,678,Morelia,71.72
24998,222,Toluca,241.21


LEFT JOIN - devuelve todos los registros de la tabla de la izquierda, aunque no tengan coincidencia.

In [24]:
connection.rollback()

print("Transacciones por cliente, aparecen aunque no tenga transacciones un cliente")
pd.read_sql(
    "SELECT " \
    "c.customer_id, " \
    "c.city, " \
    "t.amount " \
    "FROM customers c " \
    "LEFT JOIN transactions t " \
    "ON c.customer_id = t.customer_id;",
    connection
    )

Transacciones por cliente, aparecen aunque no tenga transacciones un cliente


,customer_id,city,amount
0,917,Toluca,177.94
1,383,Mérida,983.66
2,359,Cancún,128.32
3,595,Monterrey,705.96
4,909,Querétaro,164.38
...,...,...,...
24995,611,León,218.36
24996,547,León,46.10
24997,678,Morelia,71.72
24998,222,Toluca,241.21


AVG

In [25]:
connection.rollback()

print("Promedio gastado por cliente ordenando de manera descendente")
pd.read_sql(
    "SELECT " \
    "customer_id, " \
    "AVG(amount) AS average_transaction " \
    "FROM transactions " \
    "GROUP BY customer_id " \
    "ORDER BY average_transaction DESC;",
    connection
    )

Promedio gastado por cliente ordenando de manera descendente


,customer_id,average_transaction
0,500,1611.438333
1,357,1561.776250
2,261,1495.782857
3,284,1396.683846
4,807,1251.533750
...,...,...
995,113,96.636667
996,386,91.110000
997,63,90.064000
998,546,82.905000


MIN / MAX

In [26]:
connection.rollback()

print("Mínimo monto de una transacción y máximo monto de una transacción")
pd.read_sql(
    "SELECT " \
    "MIN(amount) AS smallest_transaction, " \
    "MAX(amount) AS largest_transaction " \
    "FROM transactions;",
    connection
    )

Mínimo monto de una transacción y máximo monto de una transacción


,smallest_transaction,largest_transaction
0,6.98,37437.39


DISTINCT - obtiene valores únicos.

In [27]:
connection.rollback()

print("Métodos de pago")
pd.read_sql(
    "SELECT DISTINCT method_type " \
    "FROM payment_methods;",
    connection
    )

Métodos de pago


,method_type
0,Efectivo
1,Tarjeta de débito
2,Transferencia SPEI
3,Billetera digital
4,Tarjeta de crédito


CASE WHEN - como un if and else

In [28]:
connection.rollback()

print("Clasificación de nuestros clientes por edad")
pd.read_sql(
    "SELECT " \
    "   customer_id, " \
    "   age, " \
    "   CASE " \
    "       WHEN age < 25 THEN 'Joven' " \
    "       WHEN age BETWEEN 25 AND 40 THEN 'Adulto' " \
    "       ELSE 'Adulto mayor' " \
    "   END AS age_group " \
    "FROM customers " \
    "LIMIT 3;",
    connection
    )

Clasificación de nuestros clientes por edad


,customer_id,age,age_group
0,1,58,Adulto mayor
1,2,25,Adulto
2,3,19,Joven


SUBQUERIES - consulta dentro de otra consulta

In [29]:
connection.rollback()

print("Transacciones superiores al promedio.")
pd.read_sql(
    "SELECT * " \
    "FROM transactions " \
    "WHERE amount > ( " \
    "   SELECT AVG(amount) " \
    "   FROM transactions" \
    ");",
    connection
    )

Transacciones superiores al promedio.


,transaction_id,customer_id,merchant_id,payment_method_id,amount,transaction_date
0,2,383,27,726,983.66,2025-01-01 01:51:17
1,4,595,21,1157,705.96,2025-01-01 07:00:07
2,8,182,1,345,821.10,2025-01-01 08:11:24
3,9,495,28,949,1716.87,2025-01-01 09:39:52
4,12,541,35,1039,826.16,2025-01-01 10:43:14
...,...,...,...,...,...,...
7463,24981,872,38,1709,1173.13,2026-10-20 10:16:14
7464,24985,184,34,349,899.35,2026-10-20 12:29:52
7465,24986,429,1,816,428.83,2026-10-20 12:57:30
7466,24988,774,28,1519,393.43,2026-10-20 13:53:43


CTE - crea una variable con un resultado que después se puede utilizar

In [30]:
connection.rollback()

print("Transacciones por cliente mayores a 5000")
pd.read_sql(
    "WITH customer_spending AS (" \
    "   SELECT " \
    "       customer_id, " \
    "       SUM(amount) AS total_spent " \
    "   FROM transactions " \
    "   GROUP BY customer_id" \
    ") " \
    "SELECT * " \
    "FROM customer_spending " \
    "WHERE total_spent > 5000;",
    connection
    )

Transacciones por cliente mayores a 5000


,customer_id,total_spent
0,273,9550.00
1,951,7052.16
2,70,56413.09
3,350,9460.17
4,176,5565.47
...,...,...
528,465,5235.42
529,449,11089.99
530,148,6853.95
531,790,23986.95


FUNCIONES DE FECHA

In [31]:
connection.rollback()

print("Transacciones por año")
pd.read_sql(
    "SELECT " \
    "   EXTRACT(YEAR FROM transaction_date::date) AS year," \
    "   COUNT(*) AS transactions " \
    "FROM transactions " \
    "GROUP BY EXTRACT(YEAR FROM transaction_date::date) " \
    "ORDER BY EXTRACT(YEAR FROM transaction_date::date);",
    connection
)

Transacciones por año


,year,transactions
0,2025.0,14187
1,2026.0,10813


In [32]:
connection.rollback()

print("Transacciones por mes")

pd.read_sql(
    "SELECT " \
    "   EXTRACT(MONTH FROM transaction_date::date) AS month," \
    "   COUNT(*) AS transactions " \
    "FROM transactions " \
    "GROUP BY EXTRACT(MONTH FROM transaction_date::date) " \
    "ORDER BY EXTRACT(MONTH FROM transaction_date::date);",
    connection
)

Transacciones por mes


,month,transactions
0,1.0,1914
1,2.0,2029
2,3.0,2304
3,4.0,2180
4,5.0,2595
5,6.0,2375
6,7.0,2239
7,8.0,2311
8,9.0,2112
9,10.0,1851


In [33]:
connection.rollback()

print("Ventas por año")

pd.read_sql(
    "SELECT " \
    "   EXTRACT(YEAR FROM transaction_date::date) AS YEAR," \
    "   SUM(amount) AS total_sales " \
    "FROM transactions " \
    "GROUP BY EXTRACT(YEAR FROM transaction_date::date) " \
    "ORDER BY EXTRACT(YEAR FROM transaction_date::date);",
    connection
)

Ventas por año


,year,total_sales
0,2025.0,5417529.45
1,2026.0,4052225.45


Diferencia entre WHERE y HAVING - WHERE filtra filas antes de agrupar y HAVING filtra grupos después de agrupar

OVER

In [34]:
connection.rollback()

print("Crea una columna extra con el monto máximo por cliente")

pd.read_sql(
    "SELECT " \
    "   t.transaction_id, " \
    "   c.customer_id, " \
    "   c.city, t.amount, " \
    "   MAX(t.amount) OVER (" \
    "       PARTITION BY t.customer_id" \
    "   ) AS max_amount " \
    "FROM transactions t " \
    "JOIN customers c " \
    "   ON t.customer_id = c.customer_id " \
    "ORDER BY c.customer_id, t.transaction_date;",
    connection
)

Crea una columna extra con el monto máximo por cliente


,transaction_id,customer_id,city,amount,max_amount
0,5871,1,Mérida,174.50,846.83
1,13891,1,Mérida,846.83,846.83
2,14521,1,Mérida,80.40,846.83
3,15688,1,Mérida,24.03,846.83
4,15902,1,Mérida,20.86,846.83
...,...,...,...,...,...
24995,16567,1000,Puebla,117.19,651.78
24996,18516,1000,Puebla,651.78,651.78
24997,19657,1000,Puebla,129.00,651.78
24998,19852,1000,Puebla,308.13,651.78


CONSULTA INTEGRADORA 1

In [35]:
connection.rollback()

print("Los 10 clientes que más dinero han gastado en transacciones superiores a $100, mostrando su ciudad, cantidad de transacciones y gasto total")

pd.read_sql(
    "SELECT " \
    "   c.customer_id, " \
    "   c.city," \
    "   COUNT(t.transaction_id) AS transaction_count, " \
    "   SUM(t.amount) AS total_spent, " \
    "   AVG(t.amount) AS average_transaction " \
    "FROM customers c " \
    "INNER JOIN transactions t " \
    "   ON c.customer_id = t.customer_id " \
    "WHERE t.amount > 100 " \
    "GROUP BY " \
    "   c.customer_id, " \
    "   c.city " \
    "HAVING SUM(t.amount) > 1000 " \
    "ORDER BY total_spent DESC " \
    "LIMIT 10;",
    connection
)

Los 10 clientes que más dinero han gastado en transacciones superiores a $100, mostrando su ciudad, cantidad de transacciones y gasto total


,customer_id,city,transaction_count,total_spent,average_transaction
0,28,San Luis Potosí,530,250062.05,471.815189
1,775,Ciudad de México,358,183271.67,511.932039
2,235,Tijuana,265,120793.91,455.826075
3,577,Aguascalientes,268,104941.08,391.571194
4,452,Querétaro,116,85589.15,737.837500
5,263,Ciudad de México,148,83944.29,567.191149
6,230,Veracruz,166,80932.10,487.542771
7,611,León,168,73566.42,437.895357
8,787,San Luis Potosí,143,70064.26,489.959860
9,6,Guadalajara,155,68048.96,439.025548


CONSULTA INTEGREADORA 2

In [36]:
connection.rollback()

print(
    "Contar las transacciones mayores o iguales a 1000 por ciudad, "
    "calcular su porcentaje respecto al total de transacciones "
    "de cada ciudad y obtener el monto máximo por ciudad."
)

pd.read_sql(
    "SELECT c.city, " \
    "   SUM(" \
    "       CASE " \
    "           WHEN t.amount >= 1000 THEN 1 " \
    "           ELSE 0 " \
    "       END" \
    "   ) AS num_trans_200, " \
    "   ROUND(" \
    "       100.0 * SUM(" \
    "           CASE WHEN t.amount >= 1000 THEN 1 " \
    "           ELSE 0 " \
    "       END" \
    "   ) / COUNT(*)," \
    "   1" \
    "   ) AS num_trans_200_rate, " \
    "   MAX(t.amount) AS max_amount_city " \
    "FROM customers c " \
    "INNER JOIN transactions t " \
    "   ON c.customer_id = t.customer_id " \
    "GROUP BY c.city " \
    "ORDER BY num_trans_200 DESC;",
    connection
)




Contar las transacciones mayores o iguales a 1000 por ciudad, calcular su porcentaje respecto al total de transacciones de cada ciudad y obtener el monto máximo por ciudad.


,city,num_trans_200,num_trans_200_rate,max_amount_city
0,Ciudad de México,343,7.2,19456.84
1,Guadalajara,195,6.4,14021.88
2,Monterrey,185,7.1,15381.45
3,Querétaro,116,5.8,37437.39
4,Mérida,110,6.5,12377.46
5,San Luis Potosí,107,7.0,9422.69
6,Cancún,92,7.3,10131.07
7,Toluca,92,6.3,9012.67
8,León,84,6.8,12939.59
9,Tijuana,74,6.5,9335.51


ORDEN RECOMENDADO DE LAS CONSULTAS:
1. SELECT
2. FROM
3. JOIN
4. WHERE
5. GROUP BY
6. HAVING
7. ORDER BY

## CONSULTAS TIPO ENTREVISTA

### 1. Encuentra los **10 clientes que más dinero han gastado** en total.

Resultado esperado

La consulta debe devolver:

| customer_id | city | total_spent |
|---|---|---:|
| ... | ... | ... |

El resultado debe estar ordenado del cliente que **más gastó al que menos gastó**.

In [37]:
connection.rollback()

print("10 clientes que más han gastado en total")

pd.read_sql(
    "SELECT c.customer_id, c.city, SUM(t.amount) AS total_spend " \
    "FROM customers c " \
    "JOIN transactions t " \
    "   ON c.customer_id = t.customer_id " \
    "GROUP BY " \
    "   c.customer_id," \
    "   c.city " \
    "ORDER BY total_spend DESC " \
    "LIMIT 10;",
    connection
)


10 clientes que más han gastado en total


,customer_id,city,total_spend
0,28,San Luis Potosí,258421.31
1,775,Ciudad de México,189292.23
2,235,Tijuana,125002.64
3,577,Aguascalientes,110575.15
4,452,Querétaro,88460.02
5,263,Ciudad de México,86480.68
6,230,Veracruz,84173.70
7,611,León,76440.23
8,787,San Luis Potosí,73227.92
9,6,Guadalajara,71042.88


### 2. Calcula cuánto dinero se ha generado por cada **categoría de comercio**.

Resultado esperado

La consulta debe devolver:

| category | total_sales |
|---|---:|
| ... | ... |
| ... | ... |

Ordena el resultado de **mayor a menor venta**.

In [38]:
connection.rollback()

print("Dinero que se ha generado por cada categoría")

pd.read_sql(
    "SELECT m.category, SUM(t.amount) AS total_sales " \
    "FROM merchants m " \
    "JOIN transactions t " \
    "   ON m.merchant_id = t.merchant_id " \
    "GROUP BY m.category " \
    "ORDER BY total_sales DESC;",
    connection
)

Dinero que se ha generado por cada categoría


,category,total_sales
0,Supermercado,2697348.42
1,E-commerce,1605019.93
2,Departamental,1039718.54
3,Gasolinera,970012.69
4,Hogar y ferretería,672927.41
5,Entretenimiento,562397.42
6,Restaurantes y cafés,518862.26
7,Tienda de conveniencia,420727.05
8,Farmacia,351255.59
9,Transporte,342697.55


### 3. Encuentra los clientes que hayan realizado **más de 10 transacciones**.

Resultado esperado

La consulta debe devolver:

| customer_id | transaction_count |
|---|---:|
| ... | ... |

Solo deben aparecer clientes que superen las **10 transacciones**.

Ordena el resultado de **mayor a menor número de transacciones**.

In [39]:
connection.rollback()

print("Clientes que realizaron más de 10 transacciones")

pd.read_sql(
    "SELECT customer_id, COUNT(transaction_id) AS transaction_count " \
    "FROM transactions " \
    "GROUP BY customer_id " \
    "HAVING COUNT(transaction_id) > 10",
    connection
)
# OPCIÓN CON CTE
#pd.read_sql(
#    "WITH help_count AS (" \
#    "   SELECT " \
#    "       customer_id, " \
#    "       COUNT(transaction_id) AS transaction_count " \
#    "   FROM transactions" \
#    "   GROUP BY customer_id" \
#    ")" \
#    "SELECT * " \
#    "FROM help_count " \
#    "WHERE transaction_count > 20",
#    connection
#)

Clientes que realizaron más de 10 transacciones


,customer_id,transaction_count
0,273,21
1,951,13
2,70,192
3,350,16
4,539,17
...,...,...
688,465,17
689,449,24
690,148,30
691,790,65


### 4. Calcula el **monto promedio de transacción por ciudad**, considerando únicamente transacciones mayores a **$500 MXN**.

Resultado esperado

La consulta debe devolver:

| city | average_transaction |
|---|---:|
| ... | ... |
| ... | ... |

Ordena las ciudades de **mayor a menor ticket promedio**.

In [40]:
connection.rollback()

print("Monto promedio de transacciones por ciudad considerando solo transacciones mayores a $500")

pd.read_sql(
    "WITH trans_500 AS (" \
    "   SELECT c.city, t.amount " \
    "   FROM customers c " \
    "   JOIN transactions t " \
    "       ON c.customer_id = t.customer_id " \
    "   WHERE t.amount > 500) " \
    "SELECT city, AVG(amount) AS average_transaction " \
    "FROM trans_500 " \
    "GROUP BY city " \
    "ORDER BY average_transaction DESC",
    connection
)

Monto promedio de transacciones por ciudad considerando solo transacciones mayores a $500


,city,average_transaction
0,Querétaro,1161.804101
1,Morelia,1102.523534
2,Monterrey,1073.366827
3,Ciudad de México,1068.113613
4,Cancún,1062.637782
5,Aguascalientes,1055.926235
6,Mérida,1055.363602
7,Puebla,1049.037967
8,Tijuana,1043.655620
9,Guadalajara,1040.999071


### 5. Encuentra los **10 clientes que más han gastado**, mostrando:

- `customer_id`
- `city`
- cantidad de transacciones
- gasto total
- transacción promedio

Además, **solo deben aparecer clientes cuyo gasto total sea superior a $5,000 MXN**.

Resultado esperado

La consulta debe devolver:

| customer_id | city | transaction_count | total_spent | average_transaction |
|---|---|---:|---:|---:|
| ... | ... | ... | ... | ... |

Ordena por **gasto total descendente** y muestra únicamente los **10 primeros clientes**.

In [41]:
connection.rollback()

print("10 clientes que más han gastado")

pd.read_sql(
    "SELECT c.customer_id, c.city, COUNT(t.transaction_id) AS transaction_count, SUM(t.amount) AS total_spend, AVG(t.amount) AS average_transaction " \
    "FROM customers c " \
    "JOIN transactions t " \
    "   ON c.customer_id = t.customer_id " \
    "GROUP BY " \
    "   c.customer_id, " \
    "   c.city " \
    "HAVING SUM(t.amount) > 5000 " \
    "ORDER BY total_spend DESC " \
    "LIMIT 10",
    connection
)

10 clientes que más han gastado


,customer_id,city,transaction_count,total_spend,average_transaction
0,28,San Luis Potosí,655,258421.31,394.536351
1,775,Ciudad de México,453,189292.23,417.863642
2,235,Tijuana,331,125002.64,377.651480
3,577,Aguascalientes,349,110575.15,316.834241
4,452,Querétaro,160,88460.02,552.875125
5,263,Ciudad de México,186,86480.68,464.949892
6,230,Veracruz,213,84173.70,395.181690
7,611,León,210,76440.23,364.001095
8,787,San Luis Potosí,188,73227.92,389.510213
9,6,Guadalajara,202,71042.88,351.697426


# Básico Pandas

In [42]:
import pandas as pd

customers = pd.read_sql(
    "SELECT * FROM customers",
    connection
)

merchants = pd.read_sql(
    "SELECT * FROM merchants",
    connection
)

payment_methods = pd.read_sql(
    "SELECT * FROM payment_methods",
    connection
)

transactions = pd.read_sql(
    "SELECT * FROM transactions",
    connection
)

In [43]:
print(customers.head())
print(customers.shape)
print(customers.columns)
print(customers.info())
print(customers["city"])

print(customers[
    (customers["age"] > 20)
    & (customers["age"] < 30)
])

print(customers.sort_values(
    "age",
    ascending=False
).head(10))

print(customers["age"].sum())
print(customers["age"].mean())
print(customers["age"].min())
print(customers["age"].max())
print(customers["age"].count())

   customer_id  age              city
0            1   58            Mérida
1            2   25         Monterrey
2            3   19           Tijuana
3            4   65   San Luis Potosí
4            5   35  Ciudad de México
(1000, 3)
Index(['customer_id', 'age', 'city'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  1000 non-null   int64
 1   age          1000 non-null   int64
 2   city         1000 non-null   str  
dtypes: int64(2), str(1)
memory usage: 23.6 KB
None
0                Mérida
1             Monterrey
2               Tijuana
3       San Luis Potosí
4      Ciudad de México
             ...       
995           Monterrey
996              Toluca
997              Toluca
998         Guadalajara
999              Puebla
Name: city, Length: 1000, dtype: str
     customer_id  age              city
1              2   25   